# Chapters 5 & 6: Tool Use (Function Calling) & Planning

EliteAI Academy

Define and call tools, build multi-tool agents, then generate, execute, and revise plans.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.

The code execution example runs generated Python locally. Review the code and use only trusted classroom prompts; limited builtins do not provide security isolation.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os, json, re, time, random
from typing import Any

PROVIDER = "deepseek"

if PROVIDER == "deepseek":
    import anthropic
    _client = anthropic.Anthropic(timeout=60.0, max_retries=2)
    _model  = MODEL

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.") -> str:
        r = _client.messages.create(thinking={"type": "disabled"}, 
            model=_model, max_tokens=1024, system=system,
            messages=[{"role": "user", "content": prompt}]
        )
        return r.content[0].text.strip()

elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    _gmodel = genai.GenerativeModel("gemini-2.0-flash")

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.") -> str:
        return _gmodel.generate_content(f"{system}\n\n{prompt}").text.strip()

print(f"✅ Provider: {PROVIDER}")
print(call_llm("Reply with exactly: TOOL USE READY"))

# CHAPTER 5: TOOL USE (FUNCTION CALLING)

## 1. The Problem — LLMs Are Disconnected from the World

Without tools, an LLM can only use what it learned during training. It cannot check today's weather, query a database, or send an email.

In [ ]:
questions = [
    "What is the current price of Apple (AAPL) stock?",
    "What is the weather in Lagos right now?",
    "What is 2847 * 3961?",
]

print("=== WITHOUT TOOLS (pure LLM) ===")
for q in questions:
    answer = call_llm(q)
    print(f"\n❓ {q}")
    print(f"🤖 {answer[:200]}")
    print("   ⚠️  This may be outdated, wrong, or a hallucination.")

## 2. The Tool Use Lifecycle

Define tools → Model selects a tool and arguments → Python executes it → Return the result → Model responds.

In [ ]:
lifecycle = """
╔══════════╦══════════════════════════════════════════════════════════════╗
║ Stage    ║ What Happens                                                 ║
╠══════════╬══════════════════════════════════════════════════════════════╣
║ DEFINE   ║ You describe tools as JSON schemas (name, desc, parameters)  ║
║ DECIDE   ║ LLM reads user query + tool list, decides if tool is needed  ║
║ CALL     ║ LLM outputs {"tool": "...", "args": {...}}                    ║
║ EXECUTE  ║ Your orchestration layer runs the actual Python function      ║
║ RETURN   ║ Function result is injected back into LLM context             ║
║ RESPOND  ║ LLM uses result to generate a final natural language answer   ║
╚══════════╩══════════════════════════════════════════════════════════════╝
"""
print(lifecycle)

## 3. Native Tool Calling — Anthropic SDK

The cleanest way to implement tool use: define tools as JSON schemas, pass them to the model, handle the structured tool call response.

In [ ]:
import anthropic
client = anthropic.Anthropic(timeout=60.0, max_retries=2)

def get_weather(location: str, unit: str = "celsius") -> dict:
    'Simulated weather API.'
    print(f"  🌤️  [TOOL EXECUTED] get_weather(location='{location}', unit='{unit}')")
    simulated = {
        "lagos":  {"temp": 32, "condition": "Sunny", "humidity": 78},
        "london": {"temp": 14, "condition": "Cloudy", "humidity": 85},
        "tokyo":  {"temp": 22, "condition": "Partly cloudy", "humidity": 65},
        "abuja":  {"temp": 30, "condition": "Clear", "humidity": 55},
    }
    data = simulated.get(location.lower(),
                         {"temp": 20, "condition": "Unknown", "humidity": 60})
    unit_sym = "°C" if unit == "celsius" else "°F"
    temp = data["temp"] if unit == "celsius" else round(data["temp"] * 9/5 + 32, 1)
    return {"location": location, "temperature": f"{temp}{unit_sym}",
            "condition": data["condition"], "humidity": f"{data['humidity']}%"}

def calculator(expression: str) -> dict:
    'Safe mathematical expression evaluator.'
    print(f"  🔢 [TOOL EXECUTED] calculator(expression='{expression}')")
    try:

        allowed = set('0123456789+-*/().** ')
        if not all(c in allowed for c in expression):
            return {"error": "Invalid characters in expression"}
        result = eval(expression, {"__builtins__": {}},
                      {"abs": abs, "round": round, "pow": pow})
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"error": str(e)}

def get_stock_price(ticker: str) -> dict:
    'Simulated stock price lookup.'
    print(f"  📈 [TOOL EXECUTED] get_stock_price(ticker='{ticker}')")
    prices = {"AAPL": 189.25, "GOOGL": 175.80, "MSFT": 430.50,
              "TSLA": 245.10, "AMZN": 195.60, "META": 530.20}
    ticker_upper = ticker.upper()
    if ticker_upper in prices:
        change = round(random.uniform(-3, 3), 2)
        return {"ticker": ticker_upper, "price": prices[ticker_upper],
                "change": change, "currency": "USD"}
    return {"error": f"Ticker '{ticker_upper}' not found"}

TOOL_SCHEMAS = [
    {
        "name": "get_weather",
        "description": "Get the current weather conditions for a specific location.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {"type": "string",
                             "description": "City name, e.g. 'Lagos', 'London'"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"],
                         "description": "Temperature unit"}
            },
            "required": ["location"]
        }
    },
    {
        "name": "calculator",
        "description": "Evaluate a mathematical expression precisely. Use for any calculation.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string",
                               "description": "A valid mathematical expression, e.g. '2847 * 3961'"}
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_stock_price",
        "description": "Get the current stock price for a given ticker symbol.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string",
                           "description": "Stock ticker symbol, e.g. 'AAPL', 'GOOGL'"}
            },
            "required": ["ticker"]
        }
    }
]

TOOL_REGISTRY = {
    "get_weather":    get_weather,
    "calculator":     calculator,
    "get_stock_price": get_stock_price,
}

print("✅ Tools defined:", list(TOOL_REGISTRY.keys()))

In [ ]:
def run_tool_agent(user_message: str, verbose: bool = True) -> str:
    'Full tool-use agent loop:'
    messages = [{"role": "user", "content": user_message}]

    if verbose:
        print(f"\n{'='*55}")
        print(f"USER: {user_message}")
        print(f"{'='*55}")

    for _tool_round in range(8):
        response = client.messages.create(thinking={"type": "disabled"}, 
            model=MODEL,
            max_tokens=1024,
            tools=TOOL_SCHEMAS,
            messages=messages,
            system="You are a helpful assistant. Use tools when you need real-time or calculated data."
        )

        if response.stop_reason == "end_turn":

            final_text = next(
                (block.text for block in response.content if hasattr(block, "text")), ""
            )
            if verbose:
                print(f"\n🤖 FINAL RESPONSE:\n{final_text}")
            return final_text

        elif response.stop_reason == "tool_use":

            tool_results = []

            for block in response.content:
                if block.type == "tool_use":
                    tool_name = block.name
                    tool_args = block.input

                    if verbose:
                        print(f"\n🔧 LLM CALLS TOOL: {tool_name}")
                        print(f"   Arguments: {json.dumps(tool_args, indent=2)}")

                    if tool_name in TOOL_REGISTRY:
                        result = TOOL_REGISTRY[tool_name](**tool_args)
                    else:
                        result = {"error": f"Tool '{tool_name}' not found"}

                    if verbose:
                        print(f"   Result: {json.dumps(result)}")

                    tool_results.append({
                        "type":        "tool_result",
                        "tool_use_id": block.id,
                        "content":     json.dumps(result)
                    })

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user",      "content": tool_results})

        else:
            break

    return "Stopped without a final response (maximum 8 tool rounds)."

print("=== TOOL-CALLING AGENT DEMO ===")
run_tool_agent("What's the weather like in Lagos today?")
run_tool_agent("What is 2847 multiplied by 3961?")
run_tool_agent("What's the current price of AAPL stock?")

## 4. Multi-Tool Agent — LLM Chooses the Right Tool

Give the agent ALL tools. It picks the right one based on the user's intent.

In [ ]:
multi_tool_queries = [
    "Should I buy 50 shares of TSLA at $240? What's the current price and what would that cost?",
    "Is it a good day to go outside in Tokyo? What's the weather?",
    "If I invest $10,000 at 8% annual return for 5 years, what's the final value?",
    "What's the stock price of MSFT and GOOGL?",
]

print("=== MULTI-TOOL AGENT ===")
for query in multi_tool_queries:
    print(f"\n{'─'*55}")
    run_tool_agent(query, verbose=True)

## 5. LangChain `@tool` Decorator Pattern

LangChain's `@tool` decorator auto-generates the JSON schema from the function's type hints and docstring — no manual schema writing required.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool as langchain_tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

lc_llm = ChatAnthropic(model_kwargs={"thinking": {"type": "disabled"}}, model=MODEL, temperature=0)

@langchain_tool
def lc_get_weather(location: str) -> str:
    'Get current weather for a city. Use this for any weather-related questions.'
    data = get_weather(location)
    return f"{data['condition']}, {data['temperature']}, humidity {data['humidity']}"

@langchain_tool
def lc_calculator(expression: str) -> str:
    'Evaluate a mathematical expression. Use for any arithmetic or calculations.'
    result = calculator(expression)
    return str(result.get("result", result.get("error")))

@langchain_tool
def lc_get_stock(ticker: str) -> str:
    'Get current stock price for a ticker symbol like AAPL, GOOGL, MSFT.'
    data = get_stock_price(ticker)
    if "error" in data:
        return data["error"]
    return f"{data['ticker']}: ${data['price']} (change: {data['change']:+.2f})"

@langchain_tool
def send_email(to: str, subject: str, body: str) -> str:
    'Send an email to a recipient. Use when the user asks to send an email or message.'
    print(f"  📧 [TOOL EXECUTED] send_email(to='{to}', subject='{subject}')")
    return f"Email successfully sent to {to} with subject '{subject}'"

lc_tools = [lc_get_weather, lc_calculator, lc_get_stock, send_email]

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful personal assistant. Use tools when needed to provide accurate information."),
    ("human",  "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

lc_agent    = create_tool_calling_agent(lc_llm, lc_tools, agent_prompt)
lc_executor = AgentExecutor(agent=lc_agent, tools=lc_tools, verbose=True)

lc_queries = [
    "What's the weather in London?",
    "Calculate 15% tip on a $87.50 restaurant bill.",
    "Send an email to john@example.com about our meeting tomorrow at 3pm.",
]

print("=== LangChain @tool AGENT ===")
for q in lc_queries:
    print(f"\n{'─'*55}\n❓ Query: {q}")
    result = lc_executor.invoke({"input": q})
    print(f"✅ Answer: {result['output']}")

## 6. Use Case — Financial Agent (Stock + Calculator Combined)

In [ ]:
def get_portfolio(user_id: str) -> dict:
    'Simulated portfolio database lookup.'
    print(f"  💼 [TOOL EXECUTED] get_portfolio(user_id='{user_id}')")
    portfolios = {
        "user_001": {"holdings": [{"ticker": "AAPL", "shares": 50},
                                   {"ticker": "MSFT", "shares": 30},
                                   {"ticker": "GOOGL", "shares": 10}]},
        "user_002": {"holdings": [{"ticker": "TSLA", "shares": 100},
                                   {"ticker": "META", "shares": 25}]},
    }
    return portfolios.get(user_id, {"error": f"User {user_id} not found"})

FINANCIAL_TOOLS = [
    *TOOL_SCHEMAS,
    {
        "name": "get_portfolio",
        "description": "Retrieve a user's stock portfolio from the database.",
        "input_schema": {
            "type": "object",
            "properties": {
                "user_id": {"type": "string", "description": "The user's unique ID"}
            },
            "required": ["user_id"]
        }
    }
]

FINANCIAL_REGISTRY = {**TOOL_REGISTRY, "get_portfolio": get_portfolio}

def run_financial_agent(user_message: str) -> str:
    'Financial agent with extended tool set.'
    messages = [{"role": "user", "content": user_message}]
    system = """You are a senior financial analyst assistant.
Use tools to retrieve real-time stock prices, perform calculations, and access portfolio data.
Always show your calculations and cite current prices."""

    for _tool_round in range(8):
        response = client.messages.create(thinking={"type": "disabled"}, 
            model=MODEL, max_tokens=1024,
            tools=FINANCIAL_TOOLS, messages=messages, system=system
        )
        if response.stop_reason == "end_turn":
            return next((b.text for b in response.content if hasattr(b, "text")), "")

        if response.stop_reason != "tool_use":
            return f"Model stopped with {response.stop_reason}; retry with a larger token budget."
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  🔧 Tool: {block.name} | Args: {block.input}")
                fn = FINANCIAL_REGISTRY.get(block.name, lambda **k: {"error": "not found"})
                result = fn(**block.input)
                print(f"     Result: {result}")
                tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                     "content": json.dumps(result)})
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user",      "content": tool_results})

    return "Stopped after 8 tool rounds. Simplify the request and try again."

print("=== FINANCIAL AGENT ===")
financial_queries = [
    "What is the current price of AAPL and how much would 100 shares cost?",
    "Get the portfolio for user_001 and calculate its total value using current stock prices.",
]

for q in financial_queries:
    print(f"\n{'─'*55}\n❓ {q}")
    answer = run_financial_agent(q)
    print(f"\n✅ {answer}")

## 7. Use Case — Code Execution Tool

The agent writes Python code, executes it in a sandboxed environment, and returns the result.

In [ ]:
import traceback
from io import StringIO
import sys

def execute_python_code(code: str) -> dict:
    'Execute trusted classroom code with limited builtins; this is not a security sandbox.'
    print(f"  🐍 [TOOL EXECUTED] execute_python_code")
    print(f"     Code:\n{code}")

    old_stdout = sys.stdout
    sys.stdout  = StringIO()

    try:

        safe_globals = {
            "__builtins__": {
                "print": print, "range": range, "len": len,
                "sum": sum, "max": max, "min": min, "abs": abs,
                "round": round, "enumerate": enumerate, "zip": zip,
                "list": list, "dict": dict, "set": set, "tuple": tuple,
                "sorted": sorted, "reversed": reversed, "int": int,
                "float": float, "str": str, "bool": bool, "type": type,
            },
            "math": __import__("math"),
        }
        exec(code, safe_globals)
        output = sys.stdout.getvalue()
        return {"success": True, "output": output.strip()}
    except Exception as e:
        return {"success": False, "error": str(e),
                "traceback": traceback.format_exc()}
    finally:
        sys.stdout = old_stdout

CODE_TOOL_SCHEMA = [{
    "name": "execute_python_code",
    "description": "Execute Python code to perform calculations, data processing, or analysis. Use for complex computations.",
    "input_schema": {
        "type": "object",
        "properties": {
            "code": {"type": "string",
                     "description": "Valid Python code to execute. Use print() to output results."}
        },
        "required": ["code"]
    }
}]

CODE_REGISTRY = {"execute_python_code": execute_python_code}

def run_code_agent(user_message: str) -> str:
    messages = [{"role": "user", "content": user_message}]
    system = """You are a calculator agent. When given mathematical or programming problems,
write and execute Python code to compute the answer. Always use print() to show results. The math module is already available as math; do not import modules. Available builtins include range, sum, len, min, max, list, int, and print."""

    for _tool_round in range(8):
        response = client.messages.create(thinking={"type": "disabled"}, 
            model=MODEL, max_tokens=1024,
            tools=CODE_TOOL_SCHEMA, messages=messages, system=system
        )
        if response.stop_reason == "end_turn":
            return next((b.text for b in response.content if hasattr(b, "text")), "")
        if response.stop_reason != "tool_use":
            return f"Model stopped with {response.stop_reason}; retry with a larger token budget."
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = CODE_REGISTRY[block.name](**block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                     "content": json.dumps(result)})
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user",      "content": tool_results})

    return "Stopped after 8 tool rounds. Simplify the request and try again."

print("=== CODE EXECUTION AGENT ===")
code_queries = [
    "Calculate 10 factorial.",
    "What is the sum of all prime numbers below 100?",
    "Generate the first 10 Fibonacci numbers.",
]
for q in code_queries:
    print(f"\n❓ {q}")
    ans = run_code_agent(q)
    print(f"✅ {ans}")

## 8. Tool Use Best Practices

In [ ]:
best_practices = """
╔══════════════════════════════════════╦═══════════════════════════════════════════╗
║ Good Tool Definition                 ║ Bad Tool Definition                       ║
╠══════════════════════════════════════╬═══════════════════════════════════════════╣
║ Clear, specific description          ║ Vague: "does stuff with data"             ║
║ "Get weather for a city. Returns     ║                                           ║
║  temp, condition, humidity."         ║                                           ║
╠══════════════════════════════════════╬═══════════════════════════════════════════╣
║ Parameters have types + descriptions ║ Parameters with no description            ║
║ "location: string — city name,       ║ "location: string"                        ║
║  e.g. 'London', 'Lagos'"             ║                                           ║
╠══════════════════════════════════════╬═══════════════════════════════════════════╣
║ Required vs optional clearly marked  ║ All parameters marked required            ║
╠══════════════════════════════════════╬═══════════════════════════════════════════╣
║ Returns structured dict with error   ║ Returns unstructured string or raises     ║
║ key for failures                     ║ raw exceptions                            ║
╠══════════════════════════════════════╬═══════════════════════════════════════════╣
║ Tool name is a verb + noun           ║ Tool name is ambiguous: "data", "helper"  ║
║ get_weather, send_email, run_query   ║                                           ║
╚══════════════════════════════════════╩═══════════════════════════════════════════╝

⚠️  KEY RULES:
  • Never give tools more permissions than needed (principle of least privilege)
  • Always validate and sanitise tool inputs before execution
  • Log every tool call for debugging and auditing
  • Handle tool failures gracefully — return {"error": "..."} not raw exceptions
"""
print(best_practices)

# CHAPTER 6: PLANNING

## 9. What Is Planning and When to Use It?

**Planning** = The agent formulates a multi-step sequence of actions BEFORE executing them.  
The "how" is discovered, not predefined.

```
Fixed Workflow:  A → B → C → D  (steps known in advance)
Planning Agent:  Goal → [Plan: Step 1, 2, 3, ...] → Execute each step → Result
```

In [ ]:
planning_vs_fixed = """
╔══════════════════════════════════╦════════════════════════════════════════════╗
║ Use PLANNING when...             ║ Use FIXED WORKFLOW when...                 ║
╠══════════════════════════════════╬════════════════════════════════════════════╣
║ The "how" is unknown upfront     ║ The steps are always the same              ║
║ Task is complex & multi-step     ║ Speed and predictability matter more       ║
║ Steps depend on intermediate     ║ than flexibility                           ║
║ results                          ║                                            ║
║ Environment is dynamic (can      ║ Task has low variance                      ║
║ change between steps)            ║                                            ║
║ User goal is open-ended          ║ Compliance/auditability requires exact     ║
║                                  ║ reproducibility                            ║
╠══════════════════════════════════╬════════════════════════════════════════════╣
║ EXAMPLES                                                                      ║
║ Planning: Research report, competitive analysis, employee onboarding          ║
║ Fixed:    Invoice processing, nightly DB backup, weekly report generation     ║
╚════════════════════════════════════════════════════════════════════════════════╝
"""
print(planning_vs_fixed)

## 10. Plan-Then-Execute Pattern

Stage 1: Generate a structured plan (JSON) Stage 2: Execute each step sequentially, passing results forward

In [ ]:
import inspect

def generate_plan(goal: str, context: str = "") -> list:
    'Ask the LLM to decompose a goal into an ordered list of steps.'
    prompt = f"""
    Decompose the following goal into a clear, ordered execution plan.
    
    Goal: {goal}
    {f'Context: {context}' if context else ''}
    
    Available tools: {", ".join(TOOL_REGISTRY)}. Use "llm" for a text-only step.
    
    Return ONLY a JSON array of steps. Each step must have:
    - "step": integer step number
    - "description": what this step does
    - "tool": tool name or "llm" if no tool needed
    - "depends_on": list of step numbers this depends on (empty list if none)
    
    Example: [{{"step": 1, "description": "Get weather", "tool": "get_weather", "depends_on": []}}]
    """
    raw = call_llm(prompt, system="You are a strategic planning agent. Output only valid JSON.")
    clean = re.sub(r"```[\w]*", "", raw).strip()
    return json.loads(clean)

def execute_plan(plan: list, goal: str) -> dict:
    'Execute each step in the plan sequentially.'
    step_results = {}
    print(f"\n📋 EXECUTING PLAN ({len(plan)} steps)")

    for step in plan:
        step_num  = step["step"]
        desc      = step["description"]
        tool_name = step.get("tool", "llm")

        print(f"\n[Step {step_num}] {desc} (tool: {tool_name})")

        prev_context = "\n".join(
            f"Step {s} result: {r}" for s, r in step_results.items()
        )

        if tool_name == "llm":

            result = call_llm(
                f"Goal: {goal}\nCurrent step: {desc}\nPrevious steps:\n{prev_context}\n\nExecute this step."
            )
        elif tool_name in TOOL_REGISTRY:

            args_raw = call_llm(
                f"""For step: '{desc}' using tool '{tool_name}'.
Function signature: {tool_name}{inspect.signature(TOOL_REGISTRY[tool_name])}
Previous results: {prev_context}
Use exactly the parameter names in this signature. Determine the arguments and return ONLY a JSON object of arguments.""",
                system="You extract function arguments. Return only valid JSON."
            )
            args_clean = re.sub(r"```[\w]*", "", args_raw).strip()
            args = json.loads(args_clean)
            result = TOOL_REGISTRY[tool_name](**args)
        else:
            result = f"Tool '{tool_name}' not available — skipped"

        step_results[step_num] = result
        print(f"   ✅ Result: {str(result)[:150]}")

    return step_results

GOAL = "Research Apple stock: get current price, calculate the cost of 100 shares, and draft a brief investment note."

print("=== PLAN-THEN-EXECUTE ===")
print(f"\n🎯 Goal: {GOAL}")

print("\n[STAGE 1] Generating plan...")
plan = generate_plan(GOAL)
print("\n📋 Generated Plan:")
for step in plan:
    print(f"  Step {step['step']}: {step['description']} [tool: {step.get('tool','llm')}]")

print("\n[STAGE 2] Executing plan...")
results = execute_plan(plan, GOAL)

synthesis = call_llm(
    f"Goal: {GOAL}\nAll step results:\n{json.dumps(results, indent=2, default=str)}\n\nWrite a final synthesis."
)
print(f"\n{'='*55}\nFINAL SYNTHESIS:\n{synthesis}")

## 11. Dynamic Re-Planning When a Step Fails

A hallmark of capable planning agents: when a step fails, the agent doesn't crash — it re-evaluates and adjusts the plan.

In [ ]:
def adaptive_plan_execute(goal: str, max_replans: int = 2) -> str:
    'Execute a plan with dynamic re-planning on failure.'
    plan         = generate_plan(goal)
    step_results = {}
    replan_count = 0

    print(f"\n🎯 Goal: {goal}")
    print("\n📋 Initial Plan:")
    for s in plan:
        print(f"  Step {s['step']}: {s['description']}")

    i = 0
    while i < len(plan):
        step     = plan[i]
        step_num = step["step"]
        desc     = step["description"]
        tool     = step.get("tool", "llm")

        print(f"\n[Step {step_num}] {desc}")

        simulated_fail = (step_num == 2 and replan_count == 0)

        if simulated_fail:
            print(f"  ❌ SIMULATED FAILURE: Tool '{tool}' unavailable")

            if replan_count < max_replans:
                replan_count += 1
                print(f"  🔄 RE-PLANNING (attempt {replan_count}/{max_replans})...")

                recovery_plan_raw = call_llm(
                    f"""The following step in our plan FAILED:
                    Step {step_num}: {desc} (tool: {tool})
                    
                    Original goal: {goal}
                    Completed steps so far: {json.dumps(step_results, default=str)}
                    
                    Generate a recovery plan to continue achieving the goal WITHOUT this tool.
                    Return ONLY a JSON array of replacement steps starting from step {step_num}.
                    Use only: llm, calculator, get_stock_price, get_weather""",
                    system="You are a planning agent. Return only valid JSON."
                )
                clean = re.sub(r"```[\w]*", "", recovery_plan_raw).strip()
                recovery_steps = json.loads(clean)

                plan = plan[:i] + recovery_steps
                print(f"  📋 Recovery plan generated ({len(recovery_steps)} new steps)")
                for rs in recovery_steps:
                    print(f"     Step {rs['step']}: {rs['description']}")
                continue
            else:
                step_results[step_num] = {"error": "Step failed, max re-plans reached"}
        else:

            prev = "\n".join(f"Step {k}: {v}" for k, v in step_results.items())
            if tool == "llm":
                result = call_llm(f"Goal: {goal}\nStep: {desc}\nPrevious: {prev}")
            elif tool in TOOL_REGISTRY:
                args_raw = call_llm(
                    f"Step: {desc}, function: {tool}{inspect.signature(TOOL_REGISTRY[tool])}, previous: {prev}. Use exactly these parameter names. Return JSON args only.",
                    system="Return only valid JSON args for the tool."
                )
                args = json.loads(re.sub(r"```[\w]*", "", args_raw).strip())
                result = TOOL_REGISTRY[tool](**args)
            else:
                result = f"Handled by LLM: {desc}"

            step_results[step_num] = result
            print(f"  ✅ {str(result)[:120]}")

        i += 1

    synthesis = call_llm(
        f"Goal: {goal}\nResults: {json.dumps(step_results, default=str)}\nSynthesize a final answer."
    )
    print(f"\n{'='*55}\nFINAL OUTPUT:\n{synthesis}")
    return synthesis

print("=== ADAPTIVE RE-PLANNING AGENT ===")
adaptive_plan_execute("Get the price of TSLA stock and create an investment summary.")

## 12. Research Pipeline — Planning + Parallelization + Tool Use

Combining three patterns: plan the research → gather sources in parallel → synthesise.

In [ ]:
import asyncio

async def acall_llm_async(prompt: str, system: str = "You are a helpful assistant.") -> str:
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(None, lambda: call_llm(prompt, system))

async def research_pipeline(topic: str) -> str:
    'Full research pipeline:'
    print(f"\n🔬 RESEARCH PIPELINE: {topic}")

    print("\n[Stage 1] Planning research sub-questions...")
    plan_raw = call_llm(
        f"""Break this research topic into exactly 4 focused sub-questions.
Topic: {topic}
Return ONLY a JSON array of 4 strings (the sub-questions).""",
        system="You are a research strategist. Return only valid JSON."
    )
    sub_questions = json.loads(re.sub(r"```[\w]*", "", plan_raw).strip())
    print("Sub-questions:")
    for i, q in enumerate(sub_questions, 1):
        print(f"  {i}. {q}")

    print("\n[Stage 2] Researching sub-questions in parallel...")
    t0 = time.perf_counter()

    answers = await asyncio.gather(*[
        acall_llm_async(
            f"Answer this research question in 3-4 sentences with specific facts: {q}",
            system="You are an expert researcher. Be specific and factual."
        )
        for q in sub_questions
    ])

    parallel_time = time.perf_counter() - t0
    print(f"✅ All {len(sub_questions)} sub-questions answered in {parallel_time:.2f}s")

    print("\n[Stage 3] Synthesising final report...")
    qa_pairs = "\n\n".join(
        f"Q: {q}\nA: {a}" for q, a in zip(sub_questions, answers)
    )

    report = call_llm(
        f"""Synthesise the following research findings into a structured report on: '{topic}'
        
Include: Executive Summary, Key Findings (one per sub-question), and a Conclusion.

Research Findings:
{qa_pairs}""",
        system="You are a senior research analyst writing a professional report."
    )

    return report

print("=== RESEARCH PIPELINE (Planning + Parallel + Synthesis) ===")
report = await research_pipeline("The impact of large language models on the software engineering profession")
print(f"\n{'='*55}\nFINAL RESEARCH REPORT\n{'='*55}")
print(report)

## 13. Planning with Crew AI Pattern (Pure Python)

Implementing the book's Plan → Write pattern without requiring the CrewAI library.

In [ ]:
def plan_and_write(topic: str) -> dict:
    'Implements the CrewAI planner-writer pattern:'
    print(f"\n📝 PLAN AND WRITE: {topic}")

    print("\n[Agent: Planner] Creating outline...")
    outline = call_llm(
        f"Create a 5-point bullet outline for a 200-word summary on: {topic}",
        system="You are an expert content strategist. Create clear, logical outlines."
    )
    print(f"Outline:\n{outline}")

    print("\n[Agent: Writer] Writing summary based on outline...")
    summary = call_llm(
        f"""Write a 200-word informative summary on: {topic}
        
Follow this outline exactly:
{outline}

Format your response as:
### Plan
(the outline above)

### Summary
(your 200-word summary)""",
        system="You are an expert technical writer. Write clearly and concisely."
    )

    return {"topic": topic, "outline": outline, "summary": summary}

result = plan_and_write("The importance of Reinforcement Learning in AI")
print(f"\n{'='*55}")
print(result["summary"])

## 14. Summary — Combined Rules of Thumb

In [ ]:
summary = """
╔═══════════════════════════════════════════════════════════════════════════╗
║  CHAPTER 5: TOOL USE                                                      ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  Use when: Agent needs real-time data, calculations, code execution,      ║
║            database access, or external API interactions                  ║
║  Key rule: Write tool descriptions as if explaining to a smart non-       ║
║            technical person — the LLM decides when/how to use them        ║
║  Always:   Return structured JSON from tools, handle errors gracefully    ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  CHAPTER 6: PLANNING                                                      ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  Use when: Task is complex, multi-step, and the path isn't predefined     ║
║  Skip when: Steps are always the same → use a fixed workflow instead      ║
║  Key rule: Always generate the plan as structured JSON (not prose)        ║
║            so you can iterate, validate, and re-plan programmatically     ║
║  Power move: Combine Planning + Tool Use + Parallelization for            ║
║              production-grade research and analysis agents                ║
╚═══════════════════════════════════════════════════════════════════════════╝
"""
print(summary)

## 15. Exercises

- **Exercise 1 — New Tool:** Add a `search_news(topic: str) -> dict` tool that simulates returning 3 news headlines. Register it and test it with the multi-tool agent.

- **Exercise 2 — Tool Validation:** Add input validation to `calculator` that rejects expressions longer than 200 characters and any expression containing `import`, `os`, or `sys`. Return a structured error dict.

- **Exercise 3 — Parallel Tool Calls:** Modify `run_tool_agent` to detect when the LLM requests multiple tools in one response and execute them in parallel using `concurrent.futures.ThreadPoolExecutor`.

- **Exercise 4 — Plan Validation:** Before executing a plan, add a validator step that checks: (a) all tools in the plan exist in the registry, (b) there are no circular dependencies, (c) the plan has at most 6 steps. If validation fails, re-generate the plan.

- **Exercise 5 — Deep Research Pipeline:** Extend the research pipeline (Section 12) to also generate 3 follow-up questions after the report, then answer those in a second parallel round — implementing a 2-round iterative research loop.

**Next:** Chapter 7 — Multi-Agent Orchestration